# L0H7 — Attends only to previous token, one to semantically salient (scaled up)

**Layer 0, Head 7**

## Types exhibited:
- **Previous Token Head** (full (100-90%))
- **Semantically Salient Attender** (half (60-40%))

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, ACTIVITY_LABELS,
    get_head_types, TEXT,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Attention Pattern Visualization

In [ ]:
show_head_pattern(str_tokens, cache, layer=0, head=7)

## Attention Weight Tables

Source to destination token pairs sorted by attention weight.

In [ ]:
attention = get_attention_pattern(cache, layer=0, head=7)
show_attention_tables(str_tokens, attention, top_k=25)

## Attention Metrics

In [ ]:
tm = compute_all_type_metrics(cache, str_tokens)
head_types = get_head_types(0, 7)
# Show all measurable metrics for this head
lines = []
for tid in HEAD_TYPES:
    key = (tid, 0, 7)
    if key in tm:
        lines.append(f"| {HEAD_TYPES[tid][0]} | {tm[key]:.2f}% |")
# Show non-measurable assigned types
for tid, act in head_types:
    if (tid, 0, 7) not in tm:
        lines.append(f"| {HEAD_TYPES[tid][0]} | {ACTIVITY_LABELS[act]} |")
table = "| Type | Value |\n|------|-------|\n" + "\n".join(lines)
display(Markdown(table))

In [ ]:
import importlib
import shared
importlib.reload(shared)
from shared import compute_all_type_metrics, HEAD_TYPES

tm = compute_all_type_metrics(cache, str_tokens)

# Show comma and period metrics for a few heads
print(f"{'Head':<8} {'To , %':<10} {'From , %':<10} {'Either , %':<12} {'To . %':<10} {'From . %':<10} {'Either . %':<12}")
print("-" * 72)
for layer in range(2):
    for head in range(12):
        to_c = tm[("comma_attention_to", layer, head)]
        fr_c = tm[("comma_attention_from", layer, head)]
        ei_c = tm[("comma_attention", layer, head)]
        to_p = tm[("period_attention_to", layer, head)]
        fr_p = tm[("period_attention_from", layer, head)]
        ei_p = tm[("period_attention", layer, head)]
        print(f"L{layer}H{head:<5} {to_c:>7.2f}%  {fr_c:>7.2f}%  {ei_c:>8.2f}%    {to_p:>7.2f}%  {fr_p:>7.2f}%  {ei_p:>8.2f}%")
